# Exp 41: Best vs fixed 0.5 thresholds

Card: [exp41_best_vs_fixed_05.md](../../docs/neuroslope/experiments/exp41_best_vs_fixed_05.md)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import precision_recall_curve

from training.config import CLASSES, ROOT
from training.config import VIP
from training.cv import CVResult
from analysis.plots import save_figures, use_style
from analysis.metrics import f1_at
from analysis.thresholds import best_f1

CONFIG = ROOT / "configs" / "models" / "exp29_downsample_5fold" / "resnet18.yaml"
FIXED = (0.5, 0.5, 0.5, 0.5, 0.5)  # per class, CLASSES order
FIGURE = (
    ROOT / "docs" / "figures" / "exp41_best_vs_fixed_05_pr_curves.png"
)

# dataviz reference palette, light mode
INK, BEST, FIXED_COLOR = "#52514e", "#eb6834", "#2a78d6"
use_style()

## F1 per class and Macro F1


In [ ]:
result = CVResult.load(CONFIG)  # pooled OOF rows of folds 0-4
points = {
    "best-F1": best_f1(result.labels, result.probs, CLASSES),
    "all 0.5": f1_at(result.labels, result.probs, FIXED, CLASSES),
}  # threshold, precision, recall, F1
table = pd.concat({rule: p[["threshold", "f1"]] for rule, p in points.items()}, axis=1)
table.loc["Macro F1"] = [
    float("nan") if col == "threshold" else table[(rule, col)].mean()
    for rule, col in table.columns
]
table.round(3)

## PR curves, VIP classes


In [ ]:
fig, axes = plt.subplots(1, len(VIP), figsize=(3.6 * len(VIP), 3.6), sharey=True)
for cls, ax in zip(VIP, axes):
    c = CLASSES.index(cls)
    precision, recall, _ = precision_recall_curve(
        result.labels[:, c], result.probs[:, c]
    )
    ax.plot(recall, precision, color=INK, lw=2, drawstyle="steps-post")
    for (rule, p), color in zip(points.items(), (BEST, FIXED_COLOR)):
        row = p.loc[cls]
        ax.plot(
            row["recall"],
            row["precision"],
            "o",
            ms=8,
            color=color,
            mec="#fcfcfb",
            mew=2,
            label=f"{rule}: t {row['threshold']:.3f}, F1 {row['f1']:.3f}",
        )
    ax.legend(loc="lower left", fontsize=8, frameon=False, labelcolor=INK)
    ax.set(
        title=f"{cls} ({int(result.labels[:, c].sum())} positives)",
        xlabel="recall",
        xlim=(0, 1.02),
        ylim=(0, 1.05),
    )
axes[0].set_ylabel("precision")
fig.suptitle("exp29 ResNet-18, out-of-fold rows of folds 0–4", color=INK)
fig.tight_layout()
save_figures({FIGURE.name: fig}, FIGURE.parent, dpi=150, bbox_inches="tight")
plt.show()